# 08.4 — The .mat round-trip test

How do you *prove* the Python output ([08.2](08.2_writing_mat_files_for_matlab.ipynb)) is genuinely MATLAB-compatible — without eyeballing it? A **round-trip test**: take a *real* MATLAB-produced results table, re-emit it through the Python writer, load it back, and assert every field, shape, and value survived unchanged. This is the **T4 parity gate** — the last of the project's parity tiers, guarding output-format compatibility. It runs in two levels: a pure-Python round-trip that runs in CI without MATLAB, and a true MATLAB round-trip gated behind a marker. This notebook is that gate, run live against the actual reference fixture.

This notebook covers:

* Round-trip testing: write → read back → assert identical.
* The two levels: pure-Python (CI) and true-MATLAB (`needs_matlab`).
* The reference fixture — a real 106×4×59 MATLAB `CM_Table`.
* Why a fixture-pinned schema catches silent regressions.

**Prerequisites:** [08.2 writing .mat files](08.2_writing_mat_files_for_matlab.ipynb), [00.7 pip & packaging](../00_orientation/00.7_pip_packaging_and_project_anatomy.ipynb) (pytest).

## Section 1 — What MATLAB does

There's no MATLAB *code* here — T4 is about MATLAB *consumption*. The migration plan defines the tier:

> **T4 — output-format parity:** Python output `.mat` files load cleanly into the MATLAB analysis scripts and produce the same downstream figures.

The other tiers verify *numerics* (T2 forward-pass parity, ~1e-9; the ELBO/MIL kernels). T4 verifies *format*: that `DATA_cggAllNetworkEncoderResults.m` can `load` a Python-written `CM_Table.mat` and find every field it expects, with the right types and shapes. The test enforces this by round-tripping a genuine MATLAB reference table through the Python writer and checking nothing changed.

## Section 2 — The Python concepts you need

### 2.1 — What a round-trip test is

A round-trip test checks that two operations are *inverses*: `read(write(x)) == x`. If the Python writer produces a file the reader can load back to the identical data, the writer is faithful. It's a powerful, cheap invariant — it catches any field dropped, any shape transposed, any dtype coerced, without needing a human to inspect the file. The subtlety here: the `x` we round-trip isn't arbitrary — it's a *real MATLAB table*, so passing the round-trip means the Python writer reproduces MATLAB's exact structure.

### 2.2 — Two levels of the gate

The T4 gate runs at two levels, for a reason:

| Level | Tools | Runs in CI? | Proves |
|---|---|---|---|
| **Pure-Python round-trip** | `scipy.io` write + read | ✅ always | schema/shape/value fidelity |
| **True MATLAB round-trip** | spawns MATLAB | ❌ gated (`needs_matlab`) | actual MATLAB `istable`/columns |

The pure-Python level runs on every commit (no MATLAB needed) and catches the *common* regressions — a renamed field, a transposed array. The MATLAB level (marked `needs_matlab`, deselected by default) spawns MATLAB to confirm the file promotes to a genuine `table` with the right columns — the ground truth, run when MATLAB is available. Two levels means fast feedback in CI plus a real-MATLAB backstop.

### 2.3 — The reference fixture, loaded live

In [ ]:
import numpy as np, scipy.io
from pathlib import Path

ref_path = Path("../../tests/fixtures/reference_cm_tables/CM_Table_python_struct.mat")
ref = scipy.io.loadmat(ref_path, struct_as_record=False, squeeze_me=False)["CM_Table"][0, 0]
window_keys = sorted((f for f in ref._fieldnames if f.startswith("Window_")),
                     key=lambda s: int(s.split("_")[1]))
N, D = ref.TrueValue.shape
print(f"reference CM_Table: {N} trials × {D} dims × {len(window_keys)} windows")
print(f"fields ({len(ref._fieldnames)}): DataNumber, TrueValue, Window_1..Window_{len(window_keys)}, "
      "Aggregation_Prediction, TrialConfidence, TaskConfidence")
print("→ a REAL MATLAB-produced table, converted to a scipy-loadable struct — the ground truth.")

This is a genuine result table from a real MATLAB run — 106 trials, 4 decoding dimensions, 59 analysis windows — converted once (via `table2struct`) into a scipy-loadable `.mat`. It pins the *exact* schema the Python writer must reproduce. Any drift shows up as a round-trip failure.

### 2.4 — The round-trip, live

In [ ]:
import tempfile, os
from neural_data_decoding.interop.cm_table_format import write_cm_table_mat

# Re-emit the reference data THROUGH the Python writer:
out = os.path.join(tempfile.mkdtemp(), "roundtrip.mat")
write_cm_table_mat(out,
    data_numbers=ref.DataNumber.ravel(),
    true_values=ref.TrueValue,
    window_predictions=[getattr(ref, w) for w in window_keys],
    aggregation_prediction=ref.Aggregation_Prediction,
    trial_confidence=ref.TrialConfidence,
    task_confidence=ref.TaskConfidence)

rt = scipy.io.loadmat(out, struct_as_record=False, squeeze_me=False)["CM_Table"][0, 0]

# Assert field set, shapes, and values all survived:
fields_ok = set(rt._fieldnames) == set(ref._fieldnames)
shapes_ok = all(getattr(rt, f).shape == getattr(ref, f).shape for f in ref._fieldnames)
values_ok = all(np.array_equal(getattr(rt, f), getattr(ref, f)) for f in ref._fieldnames)
print(f"field set identical?  {fields_ok}")
print(f"all shapes identical? {shapes_ok}")
print(f"all values identical? {values_ok}")
print("→ write(read) == identity across all 64 fields: the Python writer is MATLAB-faithful.")

Every one of the 64 fields round-trips exactly — field names, shapes, and values. That's the T4 pure-Python gate passing: the Python writer reproduces the MATLAB table's structure so precisely that a genuine MATLAB table, pushed through it, comes back unchanged. If someone later renamed `Aggregation_Prediction` or transposed `TaskConfidence`, this assertion would fail loudly — a schema regression caught before it reaches MATLAB.

### 2.5 — The true MATLAB round-trip (gated)

The pure-Python round-trip proves *structural* fidelity, but only MATLAB can confirm the file promotes to a real `table`. That's the `needs_matlab`-marked test: it writes a `.mat` in Python, then spawns MATLAB to `struct2table` it and inspect it via `describe_table_mat` — asserting `istable` is true, the expected columns are present, and the row count matches. It's deselected by default (`pyproject.toml` addopts `-m "not needs_matlab"`) so CI stays MATLAB-free, and run explicitly when validating against a MATLAB install. Both levels together are the complete T4 gate.

## Section 3 — The `neural_data_decoding` implementation

The round-trip assertions in `test_cm_table_parity.py`:

In [ ]:
from pathlib import Path
src = Path("../../tests/parity/test_cm_table_parity.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "def test_python_writer_round_trips_reference" in line)
# print the shape + value assertion loops
for k in range(i, min(i + 55, len(src))):
    line = src[k]
    if "assert" in line or "for f in" in line or "np.testing" in line or "roundtrip =" in line or "write_cm_table_mat" in line:
        print(f"{k + 1:4} | {line.rstrip()}")

Things to spot:

* The test loads the reference, re-emits through `write_cm_table_mat`, reloads, then asserts three things: the field set matches, every field's shape matches, and every field's values match (`np.testing.assert_array_equal`) — §2.4 in test form.
* `REQUIRED_FIELDS = {DataNumber, TrueValue, Aggregation_Prediction, TrialConfidence, TaskConfidence}` plus the `Window_k` set — the schema contract, checked explicitly so a *missing* field fails distinctly from a *wrong* one.
* Separate small tests pin the confidence shapes (`TrialConfidence (N,1)`, `TaskConfidence (N,D)`) and the `DataNumber` column orientation — the [08.2 §2.3](08.2_writing_mat_files_for_matlab.ipynb) conventions, each guarded independently.
* The MATLAB-spawning counterpart (`test_matlab_table_writer.py`) carries `pytestmark = pytest.mark.needs_matlab` and auto-skips when no MATLAB is found ([the runner](../../src/neural_data_decoding/interop/matlab_runner.py), §2.5).
* The fixtures live under `tests/fixtures/reference_cm_tables/`; they're produced once by a small MATLAB conversion script and committed, so the gate needs no MATLAB to run the pure-Python level.

## Section 4 — Hands-on exercises

### Exercise 4.1 — a regression the gate catches

Simulate a buggy writer that transposes `TaskConfidence`, and show the round-trip assertion fails (the value/shape check catches it).

In [ ]:
# Reveal:
buggy_task = ref.TaskConfidence.T          # a bug: (N,D) → (D,N)
print("correct TaskConfidence shape:", ref.TaskConfidence.shape)
print("buggy   TaskConfidence shape:", buggy_task.shape)
print("round-trip shape check would fail?", buggy_task.shape != ref.TaskConfidence.shape)
print("→ the shape assertion flags the transpose before it ever reaches MATLAB.")

### Exercise 4.2 — the round-trip is exact, not approximate

Confirm the round-trip preserves values *exactly* (`array_equal`), not just approximately — because `.mat` serialization of these dtypes is lossless, so parity is bit-exact, not tolerance-based like the numeric tiers.

In [ ]:
# Reveal:
exact = np.array_equal(rt.Aggregation_Prediction, ref.Aggregation_Prediction)
maxdiff = np.abs(rt.Aggregation_Prediction - ref.Aggregation_Prediction).max()
print(f"Aggregation_Prediction exactly equal? {exact}   (max abs diff: {maxdiff})")
print("→ unlike T2 numeric parity (~1e-9 tolerance), T4 format parity is BIT-exact — 0 difference.")

## Section 5 — Common errors

### The round-trip fails after a writer change

That's the gate doing its job — you changed the output schema. Either it's an intended change (update the reference fixture and the analysis scripts together) or a regression (revert). Never "fix" the test by loosening the assertion; the whole point is bit-exact format parity.

### The reference fixture is missing

The pure-Python level needs `tests/fixtures/reference_cm_tables/*.mat` committed. If absent, regenerate via the MATLAB conversion script, or the test errors on load. (Unlike gitignored data fixtures, these are small and committed.)

### The MATLAB-level test is "skipped"

Expected without MATLAB — it's `needs_matlab`-gated and deselected by default (§2.5). "Skipped" is not "failed." Run it explicitly (with MATLAB installed) to exercise the true round-trip.

### Comparing with a tolerance

T4 is bit-exact (Exercise 4.2), not tolerance-based. Using `assert_allclose` with an `atol` would let a real serialization bug slip through. Use `array_equal` for format parity; save tolerances for the *numeric* tiers.

### Round-trip passes but MATLAB still can't read it

The pure-Python round-trip proves scipy↔scipy fidelity, not MATLAB consumability (§2.2) — a scipy quirk could round-trip yet confuse MATLAB. That's exactly why the `needs_matlab` level exists. If MATLAB chokes, run that level to see the real error.

## Section 6 — Further reading

- [`tests/parity/test_cm_table_parity.py`](../../tests/parity/test_cm_table_parity.py) — the pure-Python round-trip.
- [`tests/parity/test_matlab_table_writer.py`](../../tests/parity/test_matlab_table_writer.py) — the `needs_matlab` MATLAB round-trip.
- [`src/neural_data_decoding/interop/matlab_runner.py`](../../src/neural_data_decoding/interop/matlab_runner.py) — how Python spawns MATLAB for the gated level.

Next up: **[08.5 Weights & Biases integration](08.5_weights_and_biases_integration.ipynb)** — the modern experiment tracker, and the honest state of its wiring in this project.